# ML-04 — Search Intelligence Data Contract (Content Refresh lane)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yowxy/Flyrank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane: **Content Refresh — "out of thousands of pages, which page should an editor fix FIRST?"** (from `w02_ml_task_framing.ipynb`). One notebook, worked in order. Simple words, honest numbers.

> Working with an AI assistant? It read `skills/README.md` and loaded `writing-data-contracts` + `flyrank/flyrank-data` for this task.

In [ ]:
# 0. Setup — runs in Colab AND locally. Installs only what is missing.
import importlib, sys, subprocess
for _pkg, _mod in [("duckdb", "duckdb"), ("huggingface_hub", "huggingface_hub"), ("scikit-learn", "sklearn")]:
    try:
        importlib.import_module(_mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)

import os
def _repo_root():
    d = os.getcwd()
    while d != "/" and not os.path.isdir(os.path.join(d, "work", "notebooks")):
        d = os.path.dirname(d)
    return d
ROOT = _repo_root()
print("repo root:", ROOT)

In [ ]:
# 0b. Warehouse connection. Token NEVER pasted in a cell: env var -> Colab Secret -> prompt.
import getpass
import duckdb
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata  # Colab only; local runs skip this
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '%s')" % HF_TOKEN.replace("'", "''"))
del HF_TOKEN  # never keep the token in memory longer than needed

REL = "hf://datasets/FlyRank/internship-warehouse"
# Mid-panel month for iteration (NOT the _sample: that is June 2026, the sealed outcome window).
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
print("connected; March partition:", FACT_MAR)

## 1. Unit of analysis + time window

**The contract, in plain words (5 answers):**

1. **One row = one content item's one day of search performance** (`report_date × client × content` on `fact_content_daily_performance`). Verified in Q1 below: that triple never repeats.
2. **Tables I use:** `fact_content_daily_performance` (month=`2026-03` partition) for everything learnable; `dim_clients` only to read each client's honest history coverage (`gsc_data_start`); `dim_content` only as join context (never as features). I do NOT use `fact_content_query_90d` here — its fixed 90-day window overlaps any recent label window, so it is leakage-prone for this task.
3. **Time window:** March 2026, split into two clean 14-day windows — features from 2026-03-01–03-14, outcome from 2026-03-15–03-28. March is a mid-panel month (panel runs 2025-01-27 → 2026-06-30). June 2026 stays sealed as the future test month; the `_sample` table (June only) is used for nothing here.
4. **What I predict / rank:** a per-page **decline proxy** — `declining = (impressions in Mar 15–28) < 0.8 × (impressions in Mar 01–14)`, keeping only pages with first-window impressions ≥ 50 (tiny denominators make ratios noise). The classifier's predicted probability becomes the editor's refresh priority score; pages are ranked by it.
5. **Deliberately excluded:** anything measured inside the outcome window (Mar 15–28) as a feature — including outcome-window impressions themselves. It is knowable only *after* the decision moment (end of Mar 14), so using it would be training on the answer.

*Decision moment: end of 2026-03-14. Every feature must be knowable then; the label is realised after 2026-03-28.*

In [ ]:
# §1 — CODE: slice row count + date span (metadata-touching, near-free). Backs answers 1–3.
con.sql(f"""
    SELECT COUNT(*) AS rows_mar,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM {FACT_MAR}
""").df()

## 2. Fields: feature / label / context / excluded

*Every field I touch, in exactly one bucket. Excluded gets a why.*

**Features (max 5 — each knowable at end of Mar 14 because it is measured Mar 01–14 only):**

1. `imp_first14` — GSC impressions, Mar 01–14. *Knowable at the decision moment because the window ends Mar 14, before any prediction is made.*
2. `clk_first14` — GSC clicks, Mar 01–14. *Knowable at the decision moment because the window ends Mar 14, before any prediction is made.*
3. `avg_pos_first14` — mean `gsc_avg_position`, Mar 01–14. *Knowable at the decision moment because the window ends Mar 14, before any prediction is made.*
4. `days_with_impr_first14` — days with ≥ 1 impression, Mar 01–14 (visibility consistency). *Knowable at the decision moment because the window ends Mar 14, before any prediction is made.*
5. `ctr_first14` — `clk_first14 / imp_first14` (snippet effectiveness, ×100 handled as a ratio here, not a percentage). *Knowable at the decision moment because both inputs are Mar 01–14 measurements.*

**Label / proxy:** `is_declining` = 1 when `imp_last14 < 0.8 × imp_first14` (Mar 15–28 vs Mar 01–14), on pages with `imp_first14 ≥ 50`. A thresholded rule on an observed outcome — a proxy for "needs a refresh", not a law of nature. Never a feature.

**Context (group / join / split only, never model inputs):** `client_hash_id`, `content_hash_id` (pseudonym IDs — for grouping and future client-holdout splits), `report_date` (for window assignment), `dim_clients.gsc_data_start` (per-client honest history start).

**Excluded (one-line why each):**

- `imp_last14` and anything else from Mar 15–28 → measured after the decision moment; it *is* the outcome window (the trap in §3 proves it).
- `fact_content_query_90d` columns (`impressions_90d`, `*_last30`) → that table's fixed 90-day window overlaps the label window; only `*_prev30`-style splits would be legal, so I leave the whole table out of this contract.
- GA4-zero-filled rows (`ga4_data_available IS NOT TRUE`) → those zeros mean "no measurement", not "no engagement"; filtered, not learned from.
- IDs as features → pseudonyms carry no portable signal; memorising them fakes skill within clients and fails on new ones.

In [ ]:
# §2 — CODE: which columns actually exist on this month's partition (backs the buckets above).
con.sql(f"SELECT column_name, data_type FROM (DESCRIBE SELECT * FROM {FACT_MAR}) ORDER BY 1").df()

## 3. Verify it with queries (grain, counts, missing values, windows)

*Three verification queries on month=2026-03, each backing a contract claim. Then the five-feature frame, then the trap.*

- **Q1 (grain):** the `(report_date, client_hash_id, content_hash_id)` triple never repeats — zero rows back means one row really is one page-day.
- **Q2 (slice counts + span):** row count, distinct pages/clients, and MIN/MAX dates of the March slice.
- **Q3 (availability):** the GA4 flag is three-valued (TRUE / FALSE / NULL), so `= FALSE` and `NOT ...` silently mishandle rows — filter with `IS TRUE` and show how many rows survive.

In [ ]:
# Q1 — GRAIN: any duplicated (report_date, client, content) triples? Empty = grain holds.
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

In [ ]:
# Q2 — SLICE: row count, population size, and date span of the March slice.
con.sql(f"""
    SELECT COUNT(*) AS rows_mar,
           COUNT(DISTINCT content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id) AS clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT_MAR}
""").df()

In [ ]:
# Q3 — AVAILABILITY: three-valued flag handled with IS TRUE (never = TRUE / NOT ...).
con.sql(f"""
    SELECT COUNT(*) AS rows_mar,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_ok_rows,
           SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_false_rows,
           SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_null_rows,
           ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_surviving
    FROM {FACT_MAR}
""").df()

### 3b. Five-feature frame (one row = one page; features Mar 01–14, label Mar 15–28)

Aggregation happens in SQL (millions of page-days in, thousands of page rows out); sklearn only ever sees the small frame. `imp_first14 ≥ 50` keeps ratios out of tiny-denominator noise.

In [ ]:
# Feature frame: past (Mar 01–14) -> future (Mar 15–28) decline proxy. Honest by construction.
frame = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN gsc_impressions ELSE 0 END) AS imp_first14,
               SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN gsc_clicks ELSE 0 END) AS clk_first14,
               AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' THEN gsc_avg_position END) AS avg_pos_first14,
               SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_with_impr_first14,
               SUM(CASE WHEN report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-28' THEN gsc_impressions ELSE 0 END) AS imp_last14
        FROM {FACT_MAR}
        GROUP BY 1, 2
        HAVING imp_first14 >= 50
    )
    SELECT *, (imp_last14 < 0.8 * imp_first14)::INT AS is_declining,
           CASE WHEN imp_first14 > 0 THEN clk_first14 * 1.0 / imp_first14 ELSE 0 END AS ctr_first14
    FROM agg
""").df()
FEATURES = ["imp_first14", "clk_first14", "avg_pos_first14", "days_with_impr_first14", "ctr_first14"]
print(f"frame: {len(frame):,} pages | declining base rate: {frame['is_declining'].mean():.3f} | features: {FEATURES}")
frame.head(5)

### 3c. The trap (on purpose, then removed)

I add ONE label-derived column — `leak_ratio_last14 = imp_last14 / imp_first14` (the warehouse analogue of notebook 02's `trend_pct`: the exact quantity the label thresholds) — train the same quick tree with and without it, watch the score jump toward perfect, then **delete it** and keep the honest number. Base rate printed next to every score, so skill is measured, not implied.

In [ ]:
# The deliberate-leak experiment: same model, same split — with vs without the outcome-window column.
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:min(k, len(labels))]].mean())

y = frame["is_declining"].to_numpy()
base = max(y.mean(), 1 - y.mean())
K = min(200, len(frame))
X_honest = frame[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
frame["leak_ratio_last14"] = frame["imp_last14"] / frame["imp_first14"]  # label-derived: the exact ratio the label thresholds
X_leaky = frame[FEATURES + ["leak_ratio_last14"]].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()

idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42, stratify=y)  # ONE split, shared by both arms
Xtr, Xte, ytr, yte = X_honest[idx_tr], X_honest[idx_te], y[idx_tr], y[idx_te]
tree_h = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xtr, ytr)
tree_l = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky[idx_tr], ytr)
acc_h, acc_l = tree_h.score(Xte, yte), tree_l.score(X_leaky[idx_te], yte)
print(f"majority-class baseline accuracy: {base:.3f} | precision@{K} of a random queue ~= {y.mean():.3f}")
print(f"HONEST (5 features)            accuracy: {acc_h:.3f} | precision@{K}: {precision_at_k(tree_h.predict_proba(Xte)[:, 1], yte, K):.3f}")
print(f"LEAKY  (+ leak_ratio_last14)  accuracy: {acc_l:.3f}  <- jumps toward perfect: it read the answer")

# The leak is removed here. Everything below uses the honest frame only.
frame.drop(columns=["leak_ratio_last14"], inplace=True)
del X_leaky, tree_l
assert "leak_ratio_last14" not in frame.columns, "leak column must be deleted from the frame"
assert "imp_last14" not in FEATURES, "leak column must not survive in the feature list"
print("leak column deleted; honest number kept: accuracy = %.3f" % acc_h)

## 4. Data limits

**One named limitation of my slice: the March slice is a survivor-biased, unbalanced-panel snapshot, not a level playing field.** Per-client history depth differs wildly across the 17-month panel (some clients have the full history, others only a few months — check `dim_clients.gsc_data_start` before trusting any cross-client comparison), and my `imp_first14 ≥ 50` floor plus the GA4 `IS TRUE` discipline deliberately drop thin-history and unmeasured pages. So this frame describes *measured, established March pages*, directionally, as decision-support for ranking — it cannot speak for new, tiny, or unmeasured pages, and June (the sealed outcome month) is untouched until validation.

In [ ]:
# §4 — CODE: per-client history depth differs (the unbalanced panel behind the limitation).
con.sql(f"""
    SELECT gsc_data_start, COUNT(*) AS clients
    FROM {DIM_CLIENTS}
    GROUP BY 1 ORDER BY 1 NULLS LAST
    LIMIT 15
""").df()

## 5. Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymised hashes aggregated; no raw query text read)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
# Self-check receipts: contract shape enforced in code, not just claimed.
assert len(FEATURES) == 5, "contract allows max five features"
assert "imp_last14" not in FEATURES and "imp_last14" not in frame[FEATURES].columns
leak_cols = [c for c in frame.columns if "last14" in c and c in FEATURES]
assert leak_cols == [], f"outcome-window columns in features: {leak_cols}"
print(f"OK: 5 features {FEATURES}; no outcome-window column among them; frame {frame.shape[0]:,} pages.")
print("OK: availability used IS TRUE (Q3); _sample/June never touched; token never printed or stored.")